# Zero-Day Pipeline v4
**April 12, 2026 · MAP v5.30 · Phase 7 data generation**

## Changes from v3 (this version) / v2 (cumulative)
- **[Fix] Circularity removed**: action is now `analyst_action`, not `oracle_action`.
  `correct = (analyst_action == oracle_action)` — analyst error is simulated, not circular.
- **[Fix] Weighted wrong-action selection**: errors favor the nearest wrong centroid,
  not uniform random. Analysts mis-escalate to investigate far more than to suppress.
- **[Add] Per-category accuracy parameters** replace META-4 v9 dependency (file never existed).
- **[Add] Seeded RNG** (numpy seed=42) — error simulation is reproducible.
- **[Change] Gate 8**: correct_rate in [0.75, 0.90], not == 1.0.
- **[Add] Gate 13**: per-category correct rates non-uniform and within bounds.
- **[Change] Phase 8**: use `--clean` flag to replace v2 + inject_errors data.

### Bug fixes applied in v4
- **[Bug01]** `probs` normalized via `numpy` before `rng.choice` — prevents rare `ValueError: probabilities do not sum to 1`.
- **[Bug02/03]** `n_done` and `n_final` division-by-zero guards in Cell 4 progress and final lines.
- **[Bug04]** P_CORRECT coverage assertion at Cell 3 startup — aborts before any API spend on mismatch.
- **[Bug05]** Per-category rate division-by-zero guard in Cell 6 report.
- **[Bug06]** `oracle_action` docstring corrected: returns raw squared distances (not sorted).
- **[Bug07]** `time.sleep(30)` before incrementing refill failure counter — transient rate-limit windows self-resolve.

## Accuracy Targets (per-category p_correct)
```
credential_access:    84%
lateral_movement:     90%   ← analysts most reliable here
data_exfiltration:    80%
malware_execution:    82%
insider_threat:       75%   ← hardest for analysts
cloud_infrastructure: 78%
Overall target:       ~81.5%
```

## DO NOT modify category list, action names, oracle functions, p_cat values, or RNG seed.
## Report numbers and stop if any gate fails.

In [ ]:
# Cell 1 -- Setup
!pip install anthropic numpy -q

import anthropic
import numpy as np
import json
import uuid
from datetime import datetime
from google.colab import userdata, drive
from pathlib import Path

ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
if not ANTHROPIC_API_KEY:
    raise SystemExit('ABORT: ANTHROPIC_API_KEY not found in Colab secrets.')
opus_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# GPT-5.4 optional fallback (used only if Opus returns empty)
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = None
if OPENAI_API_KEY:
    import openai
    gpt_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    print('GPT-5.4 fallback ENABLED.')
else:
    gpt_client = None
    print('GPT-5.4 fallback NOT available (Opus only).')

FACTOR_NAMES = [
    'travel_match', 'asset_criticality', 'threat_intel_enrichment',
    'time_anomaly',  'pattern_history',   'device_trust'
]
CANONICAL_FACTORS = set(FACTOR_NAMES)

drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/public-files/gen-ai-roi/experiments'
Path(DRIVE_BASE).mkdir(parents=True, exist_ok=True)
LOCAL_BASE = '/content'
OUT_FILE   = f'{DRIVE_BASE}/zero_day_decisions.json'

print(f'Output: {OUT_FILE}')
print(f'Local backup: {LOCAL_BASE}/zero_day_decisions.json')


In [ ]:
# Cell 2 -- Oracle Infrastructure
# Unchanged from v2 EXCEPT: oracle_action_and_correct is replaced by two functions:
#   oracle_action()            -- compute the oracle's preferred action
#   simulate_analyst_action()  -- stochastic analyst model with weighted errors
#
# Category order (Phase 7 canonical):
#   0: credential_access  1: lateral_movement  2: data_exfiltration
#   3: malware_execution  4: insider_threat    5: cloud_infrastructure
# Actions: 0=escalate  1=investigate  2=suppress  3=monitor

def build_ground_truth():
    gt = np.full((6, 4, 6), 0.5)
    gt[0, 0] = [0.8, 0.9, 0.85, 0.7,  0.75, 0.3]
    gt[0, 1] = [0.5, 0.6, 0.65, 0.5,  0.55, 0.5]
    gt[0, 2] = [0.2, 0.2, 0.15, 0.3,  0.2,  0.8]
    gt[0, 3] = [0.4, 0.4, 0.35, 0.45, 0.4,  0.6]
    gt[1, 0] = [0.9, 0.7, 0.8,  0.85, 0.8,  0.25]
    gt[1, 1] = [0.55,0.5, 0.55, 0.6,  0.5,  0.5]
    gt[1, 2] = [0.1, 0.2, 0.1,  0.15, 0.1,  0.85]
    gt[1, 3] = [0.3, 0.35,0.3,  0.35, 0.3,  0.65]
    gt[3, 0] = [0.7, 0.8, 0.9,  0.6,  0.7,  0.4]
    gt[3, 1] = [0.4, 0.5, 0.6,  0.4,  0.5,  0.55]
    gt[3, 2] = [0.2, 0.3, 0.1,  0.2,  0.15, 0.75]
    gt[3, 3] = [0.35,0.4, 0.3,  0.35, 0.35, 0.6]
    for cat in [2, 4, 5]:
        gt[cat, 0] = [0.75, 0.8,  0.75, 0.7,  0.7,  0.3]
        gt[cat, 1] = [0.5,  0.55, 0.5,  0.5,  0.5,  0.5]
        gt[cat, 2] = [0.2,  0.2,  0.2,  0.25, 0.2,  0.8]
        gt[cat, 3] = [0.35, 0.4,  0.35, 0.4,  0.35, 0.6]
    return gt


GROUND_TRUTH = build_ground_truth()
ACTION_NAMES = ['escalate', 'investigate', 'suppress', 'monitor']
N_ACTIONS    = len(ACTION_NAMES)

# Verify centroids are distinct enough to be meaningful
for cat_idx in range(6):
    dists = [
        float(np.linalg.norm(GROUND_TRUTH[cat_idx, a1] - GROUND_TRUTH[cat_idx, a2]))
        for a1 in range(N_ACTIONS) for a2 in range(a1+1, N_ACTIONS)
    ]
    if min(dists) < 0.15:
        raise SystemExit(f'ABORT: cat {cat_idx} centroids too close (min={min(dists):.3f})')
print('GT verified: all centroids distinct ✓')


def oracle_action(factor_vector, cat_idx):
    """
    Step 2: Oracle's preferred action = argmin centroid distance.
    Returns (action_idx, raw_squared_distances) where action_idx is the oracle choice.
    raw_squared_distances is the UNSORTED original list — caller indexes by action index.
    Does NOT determine what action enters the DB — analyst model does that.
    """
    f = factor_vector
    dists = [float(np.sum((f - GROUND_TRUTH[cat_idx, a])**2)) for a in range(N_ACTIONS)]
    return int(np.argmin(dists)), dists


def simulate_analyst_action(oracle_idx, dists, p_correct, rng):
    """
    Step 3: Stochastic analyst model.
    With prob p_correct: analyst agrees with oracle (correct).
    With prob 1-p_correct: analyst picks a wrong action, weighted by inverse
    centroid distance — the nearest wrong action is the most likely error.
    (Analysts err toward the 'next best' choice, not randomly.)

    Args:
        oracle_idx:  int, oracle's preferred action index
        dists:       list of 4 squared distances (same order as ACTION_NAMES)
        p_correct:   float in (0,1), per-category accuracy target
        rng:         numpy Generator (seeded, do not replace with random.random)

    Returns:
        (analyst_idx, correct, confidence)
    """
    if rng.random() < p_correct:
        analyst_idx = oracle_idx
    else:
        wrong_idxs  = [i for i in range(N_ACTIONS) if i != oracle_idx]
        # Weight by inverse distance: smaller distance → more likely error
        # Guard against d=0 (shouldn't happen with real factor vectors)
        # Bug01 fix: numpy-normalize probs to exact sum=1.0 before rng.choice.
        # Python float division can leave sum(probs) outside numpy's 1e-8 tolerance.
        inv_dists   = np.array([1.0 / max(dists[i], 1e-6) for i in wrong_idxs])
        probs       = inv_dists / inv_dists.sum()  # exact normalization
        analyst_idx = int(rng.choice(wrong_idxs, p=probs))

    correct = (analyst_idx == oracle_idx)

    # Confidence: based on distance margin of the oracle action
    sorted_dists = sorted(dists)
    margin       = sorted_dists[1] - sorted_dists[0]
    confidence   = round(min(0.95, max(0.55, 0.65 + margin * 1.5)), 2)
    if not correct:
        # Analyst confidence is lower when wrong
        confidence = round(max(0.50, confidence - 0.15), 2)

    return analyst_idx, correct, confidence


# ── Parser (shared by Opus + GPT paths, unchanged from v2) ────────────────────
def parse_fv(vec_dict):
    vals = []
    for fname in FACTOR_NAMES:
        raw = vec_dict.get(fname, 0.5)
        try:    vals.append(max(0.0, min(1.0, float(raw))))
        except: print(f'  WARNING: non-numeric {fname}={raw!r}'); vals.append(0.5)
    return np.array(vals)


def _parse_vector_response(text, n, label):
    text = text.strip().replace('```json', '').replace('```', '').strip()
    parsed = None
    try:
        r = json.loads(text)
        parsed = r if isinstance(r, list) else next(
            (v for v in r.values() if isinstance(v, list)), None)
    except json.JSONDecodeError:
        pass
    if parsed is None:
        start = text.find('[')
        if start != -1:
            depth, in_str, esc = 0, False, False
            for i, ch in enumerate(text[start:], start):
                if esc:                   esc = False; continue
                if ch == '\\' and in_str: esc = True;  continue
                if ch == '"' and not esc: in_str = not in_str; continue
                if in_str: continue
                if ch == '[':  depth += 1
                elif ch == ']':
                    depth -= 1
                    if depth == 0:
                        try:    parsed = json.loads(text[start:i+1])
                        except: pass
                        break
    if parsed is None:
        print(f'  WARNING [{label}]: parse failed: {text[:100]}')
        return []
    if len(parsed) != n:
        print(f'  WARNING [{label}]: requested {n}, got {len(parsed)}')
    cleaned = [{k: v for k, v in vec.items() if k in CANONICAL_FACTORS}
               for vec in parsed if isinstance(vec, dict)]
    bad = sum(1 for v in parsed if not isinstance(v, dict))
    if bad: print(f'  WARNING [{label}]: {bad} non-dict entries dropped')
    for idx, vec in enumerate(cleaned):
        missing = CANONICAL_FACTORS - set(vec.keys())
        if missing: print(f'  WARNING [{label}]: vec[{idx}] missing {missing} (using 0.5)')
    return cleaned


COLD_START_PROMPT = (
    'Generate exactly {n} factor vectors for SOC triage alerts.\n'
    'Context: Cold-start deployment. Vary all six factors widely across [0.05, 0.95]. No clustering.\n\n'
    'Examples (do not copy):\n'
    '[\n'
    '  {{"travel_match": 0.72, "asset_criticality": 0.85, "threat_intel_enrichment": 0.14,\n'
    '   "time_anomaly": 0.61, "pattern_history": 0.08, "device_trust": 0.79}},\n'
    '  {{"travel_match": 0.12, "asset_criticality": 0.45, "threat_intel_enrichment": 0.57,\n'
    '   "time_anomaly": 0.23, "pattern_history": 0.44, "device_trust": 0.88}}\n'
    ']\n\n'
    'Return a JSON array of exactly {n} objects. '
    'Each: {{"travel_match": float, "asset_criticality": float, '
    '"threat_intel_enrichment": float, "time_anomaly": float, '
    '"pattern_history": float, "device_trust": float}}. '
    'All values 0.0-1.0. Each vector distinct. No explanation.'
)


def _generate_opus(n):
    prompt  = COLD_START_PROMPT.replace('{n}', str(n))
    max_tok = min(max(500, n * 140), 4096)
    try:
        r = opus_client.messages.create(
            model='claude-opus-4-6', max_tokens=max_tok,
            system='Output a valid JSON array only. No explanation. No markdown.',
            messages=[{'role': 'user', 'content': prompt}]
        )
    except Exception as e:
        print(f'  WARNING [OPUS]: API error: {e}')
        return []
    if not r.content:
        print('  WARNING [OPUS]: empty content')
        return []
    block = r.content[0]
    if not hasattr(block, 'text') or getattr(block, 'type', 'text') != 'text':
        print(f'  WARNING [OPUS]: unexpected block type {getattr(block, "type", type(block))}')
        return []
    return _parse_vector_response(block.text, n, 'OPUS')


def _generate_gpt(n):
    if gpt_client is None:
        return []
    prompt  = COLD_START_PROMPT.replace('{n}', str(n))
    max_tok = min(max(500, n * 140), 4096)
    try:
        r = gpt_client.chat.completions.create(
            model='gpt-5.4', max_completion_tokens=max_tok,
            messages=[
                {'role': 'system', 'content': 'Output a valid JSON array only. No explanation. No markdown.'},
                {'role': 'user',   'content': prompt}
            ]
        )
    except Exception as e:
        print(f'  WARNING [GPT-5.4]: API error: {e}')
        return []
    if not r.choices:
        print('  WARNING [GPT-5.4]: empty choices')
        return []
    return _parse_vector_response(r.choices[0].message.content or '', n, 'GPT-5.4')


def generate_factor_vectors(n):
    vecs = _generate_opus(n)
    if not vecs:
        if gpt_client is not None:
            print('  [FALLBACK] Opus empty -- retrying with GPT-5.4')
            vecs = _generate_gpt(n)
        if not vecs:
            print('  WARNING: both Opus and GPT-5.4 returned empty')
    return vecs


print('Oracle infrastructure ready.')
print(f'  Actions: {dict(enumerate(ACTION_NAMES))}')

# Quick self-test: oracle_action + simulate_analyst_action consistency
_rng_test = np.random.default_rng(0)
_f_test   = np.array([0.78, 0.85, 0.80, 0.68, 0.72, 0.32])
_oidx, _dists = oracle_action(_f_test, 0)
_aidx, _corr, _conf = simulate_analyst_action(_oidx, _dists, 1.0, _rng_test)
assert _aidx == _oidx and _corr is True,  'Self-test FAIL: p=1.0 should always be correct'
_aidx2, _corr2, _ = simulate_analyst_action(_oidx, _dists, 0.0, _rng_test)
assert _aidx2 != _oidx and _corr2 is False, 'Self-test FAIL: p=0.0 should always be wrong'
print('Self-test passed: p=1.0→correct, p=0.0→wrong ✓')


In [ ]:
# Cell 3 -- Constants

CATEGORIES = [
    # (cat_name, abbrev, severity, gt_idx)
    ('credential_access',    'CA', 'high',     0),
    ('lateral_movement',     'LM', 'high',     1),
    ('data_exfiltration',    'DE', 'critical', 2),
    ('malware_execution',    'ME', 'critical', 3),
    ('insider_threat',       'IT', 'medium',   4),
    ('cloud_infrastructure', 'CI', 'medium',   5),
]

# Per-category analyst accuracy targets.
# DO NOT modify without roadmap approval -- changes output distribution.
# Keyed by cat_name for readability and to prevent index-order bugs.
P_CORRECT = {
    'credential_access':    0.84,
    'lateral_movement':     0.90,  # analysts most reliable
    'data_exfiltration':    0.80,
    'malware_execution':    0.82,
    'insider_threat':       0.75,  # hardest for analysts
    'cloud_infrastructure': 0.78,
}

# Bug04 fix: assert P_CORRECT and CATEGORIES are in sync before any API spend.
_cats_from_categories = {c[0] for c in CATEGORIES}
_cats_from_p_correct  = set(P_CORRECT.keys())
if _cats_from_categories != _cats_from_p_correct:
    _missing = _cats_from_categories - _cats_from_p_correct
    _extra   = _cats_from_p_correct   - _cats_from_categories
    raise SystemExit(
        f'ABORT: CATEGORIES/P_CORRECT mismatch.\n'
        f'  Missing from P_CORRECT: {_missing}\n'
        f'  Extra in P_CORRECT:     {_extra}'
    )
print(f'P_CORRECT coverage check ✓ (all {len(CATEGORIES)} categories covered)')

CANONICAL_CATS = {c[0] for c in CATEGORIES}
CANONICAL_ACTS = {'escalate', 'investigate', 'suppress', 'monitor'}

# Seeded RNG -- DO NOT change seed (seed=42 → reproducible error simulation)
RNG = np.random.default_rng(42)

DAYS                = 90
DECISIONS_PER_GROUP = 9
BUFFER_SIZE         = 50
SAVE_EVERY          = 54    # every 9 days × 6 categories
PROGRESS_EVERY      = 30
BASE_TIMESTAMP      = 1741046400000  # 2025-03-04 00:00:00 UTC in ms

CORRECT_RATE_LO = 0.75
CORRECT_RATE_HI = 0.90

total_groups    = DAYS * len(CATEGORIES)
total_decisions = total_groups * DECISIONS_PER_GROUP
total_alerts    = total_groups

expected_overall = sum(
    P_CORRECT[cat] * DAYS * DECISIONS_PER_GROUP
    for cat, *_ in CATEGORIES
) / total_decisions

print(f'Volume:   {DAYS}d × {len(CATEGORIES)}cat × {DECISIONS_PER_GROUP}dec = {total_decisions} decisions')
print(f'Alerts:   {total_alerts}')
print(f'Expected overall accuracy: {expected_overall:.1%}')
print(f'Gate 8 window: [{CORRECT_RATE_LO:.0%}, {CORRECT_RATE_HI:.0%}]')
print()
print('Per-category accuracy targets:')
n_per_cat = DAYS * DECISIONS_PER_GROUP
for cat, abbrev, sev, _ in CATEGORIES:
    p   = P_CORRECT[cat]
    exp_correct = int(n_per_cat * p)
    print(f'  {cat:<25}: p={p:.0%}  (~{exp_correct}/{n_per_cat} correct)')


In [ ]:
# Cell 4 -- Generation Loop
# v3 change: oracle_action() + simulate_analyst_action() replace direct oracle assignment.
# action = analyst_action (not oracle_action).
# correct = (analyst_action == oracle_action).
# DO NOT modify the oracle functions or p_cat values here -- change Cell 3 or Cell 2.

all_alerts    = []
all_decisions = []
factor_buffer = []
consecutive_refill_failures = 0
CONSECUTIVE_FAIL_LIMIT = 5
groups_done   = 0

print(f'Starting: {total_groups} groups, {total_decisions} decisions...')
print()

for day in range(1, DAYS + 1):
    timestamp_ms = BASE_TIMESTAMP + (day - 1) * 86_400_000

    for cat_name, abbrev, severity, cat_idx in CATEGORIES:
        p_cat    = P_CORRECT[cat_name]   # per-category accuracy target
        alert_id = f'SYN-{abbrev}-D{day:03d}-001'

        all_alerts.append({
            'alert_id':        alert_id,
            'category':        cat_name,
            'severity':        severity,
            'alert_type':      cat_name,
            'timestamp_epoch': timestamp_ms,
            'origin':          'zero_day_synthetic',
            'source_location': 'synthetic',
            'user_id':         '',
            'status':          'decided',
        })

        for seq in range(DECISIONS_PER_GROUP):

            if not factor_buffer:
                factor_buffer = generate_factor_vectors(BUFFER_SIZE)
                if not factor_buffer:
                    # Bug07 fix: sleep before counting failure — lets rate-limit windows clear.
                    # Re-running Cell 4 resets all lists, so an unnecessary abort costs
                    # significant API spend and time.
                    import time
                    time.sleep(30)
                    consecutive_refill_failures += 1
                    print(f'  WARNING: refill failure #{consecutive_refill_failures} '
                          f'at day={day} cat={cat_name} seq={seq} (slept 30s)')
                    if consecutive_refill_failures >= CONSECUTIVE_FAIL_LIMIT:
                        partial = {
                            'status': 'aborted',
                            'groups_done': groups_done,
                            'alerts': all_alerts,
                            'decisions': all_decisions,
                        }
                        abort_path = f'{LOCAL_BASE}/zero_day_aborted_{groups_done}.json'
                        with open(abort_path, 'w') as fh:
                            json.dump(partial, fh)
                        raise SystemExit(
                            f'ABORT: {CONSECUTIVE_FAIL_LIMIT} consecutive refill failures. '
                            f'Partial data at {abort_path}.')
                    continue
                else:
                    consecutive_refill_failures = 0

            vec_dict  = factor_buffer.pop(0)
            f_arr     = parse_fv(vec_dict)

            # v3 core: separate oracle from analyst
            o_idx, dists             = oracle_action(f_arr, cat_idx)
            a_idx, correct, conf     = simulate_analyst_action(o_idx, dists, p_cat, RNG)

            all_decisions.append({
                'decision_id':     f'SYN-DEC-{uuid.uuid4().hex[:8]}',
                'alert_id':        alert_id,
                'category':        cat_name,
                'action':          ACTION_NAMES[a_idx],    # analyst_action, not oracle_action
                'factor_vector':   [round(x, 4) for x in f_arr.tolist()],
                'confidence':      conf,
                'correct':         correct,                # (analyst == oracle)
                'outcome':         'correct' if correct else 'incorrect',
                'timestamp_epoch': timestamp_ms,
                'origin':          'zero_day_synthetic',
                'source_id':       'synthetic',
                'user_id':         '',
            })

        groups_done += 1

        if groups_done % PROGRESS_EVERY == 0:
            n_done  = len(all_decisions)
            n_corr  = sum(1 for d in all_decisions if d['correct'])
            pct     = groups_done / total_groups * 100
            acc_str = f'{n_corr/n_done:.1%}' if n_done else 'n/a'
            print(f'  [{pct:.0f}%] {groups_done}/{total_groups} groups  '
                  f'{n_done} decisions  acc={acc_str}  day={day}')

        if groups_done % SAVE_EVERY == 0:
            interim = {
                'status': 'interim', 'groups_done': groups_done,
                'alerts': all_alerts, 'decisions': all_decisions,
            }
            local_chk = f'{LOCAL_BASE}/zero_day_interim_{groups_done}.json'
            with open(local_chk, 'w') as fh:
                json.dump(interim, fh)
            try:
                with open(f'{DRIVE_BASE}/zero_day_interim_{groups_done}.json', 'w') as fh:
                    json.dump(interim, fh)
                print(f'  [SAVE] {groups_done} groups / {len(all_decisions)} decisions (local + Drive)')
            except Exception as e:
                print(f'  [SAVE] {groups_done} groups / {len(all_decisions)} decisions '
                      f'(local only -- Drive: {e})')

n_final = len(all_decisions)
n_corr  = sum(1 for d in all_decisions if d['correct'])
print(f'\nGeneration complete: {n_final} decisions, {len(all_alerts)} alerts.')
acc_final = f'{n_corr/n_final:.1%}' if n_final else 'n/a (0 decisions generated)'
print(f'Overall accuracy: {acc_final} (target {CORRECT_RATE_LO:.0%}–{CORRECT_RATE_HI:.0%})')
print(f'Buffer remaining: {len(factor_buffer)} (discarded).')


In [ ]:
# Cell 5 -- Validation Gates (13 gates)
# Gate 8 change: correct_rate in [0.75, 0.90] (was == 1.0 in v2)
# Gate 13 added: per-category correct rates non-uniform and within per-cat bounds

from collections import Counter

TARGET_DECISIONS = total_decisions
TARGET_ALERTS    = total_alerts
REQUIRED_DEC_FIELDS = [
    'decision_id', 'alert_id', 'category', 'action', 'factor_vector',
    'confidence', 'correct', 'outcome', 'timestamp_epoch', 'origin',
]
REQUIRED_ALT_FIELDS = ['alert_id', 'category', 'severity', 'timestamp_epoch', 'origin']

gates = []
n_dec = len(all_decisions)

# G1: Decision count ±5
g1 = abs(n_dec - TARGET_DECISIONS) <= 5
gates.append(('G1: Decision count', g1,
              f'{n_dec} (target {TARGET_DECISIONS}, delta {abs(n_dec-TARGET_DECISIONS)})'))

# G2: Zero nulls
null_fields = []
for i, d in enumerate(all_decisions):
    for fld in REQUIRED_DEC_FIELDS:
        if d.get(fld) is None: null_fields.append(f'dec[{i}].{fld}')
for i, a in enumerate(all_alerts):
    for fld in REQUIRED_ALT_FIELDS:
        if a.get(fld) is None: null_fields.append(f'alt[{i}].{fld}')
g2 = len(null_fields) == 0
gates.append(('G2: Zero nulls', g2,
              'PASS' if g2 else f'{len(null_fields)} nulls: {null_fields[:5]}'))

# G3: factor_vector 6 elements, all [0,1]
bad_fv = []
for i, d in enumerate(all_decisions):
    fv = d.get('factor_vector', [])
    if not isinstance(fv, list) or len(fv) != 6:
        bad_fv.append(f'[{i}]: len={len(fv) if isinstance(fv,list) else type(fv).__name__}')
    elif any(not (0.0 <= v <= 1.0) for v in fv):
        bad_fv.append(f'[{i}]: out-of-range')
g3 = len(bad_fv) == 0
gates.append(('G3: Factor vectors', g3,
              'PASS' if g3 else f'{len(bad_fv)} bad: {bad_fv[:3]}'))

# G4: action in canonical set
bad_act = [d['action'] for d in all_decisions if d.get('action') not in CANONICAL_ACTS]
g4 = len(bad_act) == 0
gates.append(('G4: Action validity', g4,
              'PASS' if g4 else f'{len(bad_act)} invalid: {set(bad_act)}'))

# G5: category in canonical set
bad_cat = [d['category'] for d in all_decisions if d.get('category') not in CANONICAL_CATS]
g5 = len(bad_cat) == 0
gates.append(('G5: Category validity', g5,
              'PASS' if g5 else f'{len(bad_cat)} invalid: {set(bad_cat)}'))

# G6: every decision.alert_id matches an alert
alert_id_set = {a['alert_id'] for a in all_alerts}
orphan_decs  = [d['decision_id'] for d in all_decisions if d.get('alert_id') not in alert_id_set]
g6 = len(orphan_decs) == 0
gates.append(('G6: No orphan decisions', g6,
              'PASS' if g6 else f'{len(orphan_decs)} orphans'))

# G7: alert count ±5
n_alt = len(all_alerts)
g7 = abs(n_alt - TARGET_ALERTS) <= 5
gates.append(('G7: Alert count', g7, f'{n_alt} (target {TARGET_ALERTS})'))

# G8 (v3 change): overall correct_rate in [0.75, 0.90] — NOT == 1.0
n_correct    = sum(1 for d in all_decisions if d.get('correct') is True)
correct_rate = n_correct / n_dec if n_dec else 0.0
g8 = CORRECT_RATE_LO <= correct_rate <= CORRECT_RATE_HI
gates.append(('G8: Overall correct rate', g8,
              f'{correct_rate:.3f} (expected [{CORRECT_RATE_LO:.2f}, {CORRECT_RATE_HI:.2f}])'))

# G9: factor vector variance > 0.02/factor
all_fv_arr   = np.array([d['factor_vector'] for d in all_decisions])
per_fac_var  = all_fv_arr.var(axis=0)
g9 = bool(all(v > 0.02 for v in per_fac_var))
gates.append(('G9: Factor variance', g9,
              ' '.join(f'{fn[:8]}={v:.4f}' for fn, v in zip(FACTOR_NAMES, per_fac_var))))

# G10: decision_id uniqueness
dec_ids      = [d['decision_id'] for d in all_decisions]
dec_id_dupes = len(dec_ids) - len(set(dec_ids))
g10 = dec_id_dupes == 0
gates.append(('G10: decision_id unique', g10,
              'PASS' if g10 else f'{dec_id_dupes} duplicates'))

# G11: alert_id uniqueness
alt_ids      = [a['alert_id'] for a in all_alerts]
alt_id_dupes = len(alt_ids) - len(set(alt_ids))
g11 = alt_id_dupes == 0
gates.append(('G11: alert_id unique', g11,
              'PASS' if g11 else f'{alt_id_dupes} duplicates'))

# G12: every alert has exactly DECISIONS_PER_GROUP linked decisions
dec_per_alert    = Counter(d['alert_id'] for d in all_decisions)
bad_alert_counts = {
    a['alert_id']: dec_per_alert.get(a['alert_id'], 0)
    for a in all_alerts
    if dec_per_alert.get(a['alert_id'], 0) != DECISIONS_PER_GROUP
}
g12 = len(bad_alert_counts) == 0
g12_detail = (f'PASS (all {n_alt} alerts have {DECISIONS_PER_GROUP} decisions)' if g12
              else f'{len(bad_alert_counts)} alerts with wrong count -- refill failures likely')
gates.append(('G12: Per-alert dec count', g12, g12_detail))

# G13 (v3 new): per-category correct rates non-uniform and within per-cat bounds
# Tolerance: ±6pp from target (stochastic variation over 810 decisions per category)
CAT_TOLERANCE   = 0.06
cat_gate_detail = []
cat_gate_ok     = True
for cat_name, *_ in CATEGORIES:
    cat_decs   = [d for d in all_decisions if d['category'] == cat_name]
    if not cat_decs:
        cat_gate_detail.append(f'{cat_name}: NO DECISIONS')
        cat_gate_ok = False
        continue
    cat_rate   = sum(1 for d in cat_decs if d['correct']) / len(cat_decs)
    target     = P_CORRECT[cat_name]
    in_range   = abs(cat_rate - target) <= CAT_TOLERANCE
    if not in_range:
        cat_gate_ok = False
    cat_gate_detail.append(
        f'{cat_name[:10]}:{cat_rate:.1%}(tgt {target:.0%},{"OK" if in_range else "OUT"})')
# Also check non-uniformity: rates must not all be equal (max-min > 0.03)
cat_rates_vals = [
    sum(1 for d in all_decisions if d['category']==c and d['correct'])
    / max(sum(1 for d in all_decisions if d['category']==c), 1)
    for c, *_ in CATEGORIES
]
spread = max(cat_rates_vals) - min(cat_rates_vals)
if spread < 0.03:
    cat_gate_ok = False
    cat_gate_detail.append(f'UNIFORM (spread={spread:.3f} < 0.03)')
else:
    cat_gate_detail.append(f'spread={spread:.3f}')
g13 = cat_gate_ok
gates.append(('G13: Per-cat correct rates', g13,
              '  '.join(cat_gate_detail)))

# ── Report ────────────────────────────────────────────────────────────────────
print('=' * 68)
print('VALIDATION GATES')
print('=' * 68)
all_pass = True
for name, passed, detail in gates:
    status = 'PASS' if passed else 'FAIL'
    if not passed: all_pass = False
    print(f'  [{status}] {name}: {detail}')
print()
if all_pass:
    print('ALL 13 GATES PASS -- proceed to Cell 6 (save)')
else:
    print('GATE(S) FAILED -- DO NOT SAVE. Report to roadmap session.')
    raise SystemExit('Validation failed.')

print('\nAction distribution:')
for act, cnt in Counter(d['action'] for d in all_decisions).most_common():
    print(f'  {act:15s}: {cnt} ({cnt/n_dec*100:.1f}%)')
print('\nPer-category accuracy:')
for cat_name, *_ in CATEGORIES:
    cat_decs = [d for d in all_decisions if d['category'] == cat_name]
    rate     = sum(1 for d in cat_decs if d['correct']) / len(cat_decs)
    print(f'  {cat_name:<25}: {rate:.1%}  (target {P_CORRECT[cat_name]:.0%})')
print(f'  {"OVERALL":<25}: {correct_rate:.1%}  (target {CORRECT_RATE_LO:.0%}–{CORRECT_RATE_HI:.0%})')


In [ ]:
# Cell 6 -- Save + Report-back
# Only runs if Cell 5 passed all 13 gates. v4 fixes: Bug01-07.
# Phase 8: use seed_zero_day.py --live --clean to replace v2 + inject_errors data.

output = {
    'metadata': {
        'generator':           'zero_day_pipeline_v4',
        'generated':           datetime.now().isoformat(),
        'total_alerts':        len(all_alerts),
        'total_decisions':     len(all_decisions),
        'days_simulated':      DAYS,
        'decisions_per_group': DECISIONS_PER_GROUP,
        'ground_truth_source': 'GROUND_TRUTH_P1 canonical centroids (oracle_separation_v8)',
        'category_order':      [c[0] for c in CATEGORIES],
        'action_order':        ACTION_NAMES,
        'p_correct_by_cat':    P_CORRECT,
        'rng_seed':            42,
        'base_timestamp_ms':   BASE_TIMESTAMP,
        'overall_correct_rate':round(correct_rate, 4),
        'origin':              'zero_day_synthetic',
        'notes': (
            'Factor vectors: Opus cold_start oracle separation (GPT-5.4 fallback if needed). '
            'action = analyst_action (stochastic, per-category p_correct). '
            'correct = (analyst_action == oracle_action). '
            'Weighted wrong-action selection (inverse centroid distance). '
            'RNG seed=42. Phase 7 v3, MAP v5.30.'
        ),
    },
    'alerts':    all_alerts,
    'decisions': all_decisions,
}

local_path = f'{LOCAL_BASE}/zero_day_decisions.json'
with open(local_path, 'w') as f:
    json.dump(output, f, indent=2)
local_kb = Path(local_path).stat().st_size // 1024
print(f'Saved locally: {local_path}  ({local_kb} KB)')

with open(OUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)
drive_kb = Path(OUT_FILE).stat().st_size // 1024
print(f'Saved to Drive: {OUT_FILE}  ({drive_kb} KB)')

with open(OUT_FILE) as f:
    verify = json.load(f)
assert verify['metadata']['total_decisions'] == len(all_decisions), 'Drive save mismatch'
assert verify['metadata']['generator'] == 'zero_day_pipeline_v4', 'Generator tag mismatch'
print(f'Drive file verified: {verify["metadata"]["total_decisions"]} decisions  ✓')

print()
print('=' * 62)
print('PHASE 7 REPORT-BACK -- PASTE TO ROADMAP SESSION')
print('=' * 62)
print(f'  Generator       : zero_day_pipeline_v4')
print(f'  Alerts          : {len(all_alerts)}')
print(f'  Decisions       : {len(all_decisions)}')
print(f'  Days simulated  : {DAYS}')
print(f'  Overall correct : {correct_rate:.1%}  (target {CORRECT_RATE_LO:.0%}-{CORRECT_RATE_HI:.0%})')
print(f'  All 13 gates    : PASS')
print(f'  File size       : {drive_kb} KB')
print(f'  RNG seed        : 42 (reproducible)')
print()
print('Per-category accuracy (actual vs target):')
for cat_name, *_ in CATEGORIES:
    cat_decs = [d for d in all_decisions if d['category'] == cat_name]
    rate     = (sum(1 for d in cat_decs if d['correct']) / len(cat_decs)
                if cat_decs else 0.0)  # Bug05: guard empty category
    rate_str = f'{rate:.1%}' if cat_decs else 'N/A (no decisions)'
    print(f'  {cat_name:<25}: {rate_str} (tgt {P_CORRECT[cat_name]:.0%})')
print()
print('DRIVE -> REPO SYNC NEEDED:')
print(f'  New files: zero_day_decisions.json')
print(f'  Drive path: {DRIVE_BASE}/')
print(f'  Action: cp to backend/support/setup/zero_day_decisions.json')
print(f'  Then: python support/setup/seed_zero_day.py --dry-run')
print(f'  Then: python support/setup/seed_zero_day.py --live --clean')
print(f'  (--clean replaces v2 + inject_errors data)')
print('=' * 62)
